# Tugas Kelompok 2 — Evaluasi Otomatis & Analisis Error MT (EN → ID)

Pipeline: **FLORES-200 EN–ID pairs → pre-trained NMT (MarianMT / NLLB) → BLEU-1…4, chrF, chrF++, TER (+BERTScore) → n-gram analysis → worst-sentence error analysis → report**.

Run top to bottom. On **Google Colab** use *Runtime → GPU* and run the setup cell first.

## 0. Setup

In [1]:
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("translate_machine"):
    subprocess.run(["git", "clone", "https://github.com/Mutaib-yye/translate_machine"], check=True)
    os.chdir("translate_machine")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # run from the repo root
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: /content/translate_machine


## 1. Configuration

In [2]:
import os
N_SENTENCES = 200          # prof: every member may choose; 0 = use all 2009 pairs
SEED        = 42
MODELS      = ["marian", "nllb"]   # "marian" = Helsinki-NLP/opus-mt-en-id, "nllb" = facebook/nllb-200-distilled-600M
NUM_BEAMS   = 4
BATCH_SIZE  = 16
WORST_K     = 10           # >= 5 required by the assignment
USE_BERTSCORE = False      # extra metric (downloads xlm-roberta); set True for the bonus
BOOTSTRAP   = 300          # resamples for 95% confidence intervals (0 = skip)
DRY_RUN     = os.environ.get("MT_DRY_RUN") == "1"   # offline smoke test only -> NOT real results
OUT = "results_dryrun" if DRY_RUN else "results"
os.makedirs(OUT, exist_ok=True); os.makedirs("report", exist_ok=True)

## 2. Data — EN→ID parallel sentences (FLORES-200)
Mojibake in the references (e.g. `BarÃ§a`) is repaired on load.

In [3]:
import pandas as pd
from src import mt_eval as m
pd.set_option("display.max_colwidth", 140, "display.width", 200)
full = m.load_pairs()
df = m.sample_pairs(full, N_SENTENCES, SEED)
print(f"{len(full)} pairs available, using {len(df)}")
srcs, refs = df.source.tolist(), df.reference.tolist()
df.head()

2009 pairs available, using 200


,id,source,reference
0,30,Iraq's Abu Ghraib prison has been set alight during a riot.,Penjara Abu Ghraib Irak dibakar saat terjadi kerusuhan.
1,57,"""If it becomes commercial, we should have it. That is, there's no in-principle objection to nuclear energy"" Mr Costello said.","""Jika itu nanti komersil, kita harus memilikinya. Artinya, pada dasarnya tidak ada keberatan terhadap energi nuklir"" kata Tuan Costello."""
2,66,Protesters hope to collect a petition of 1.2 million signatures to present to the National Congress in November.,"Para pengunjuk rasa berharap dapat mengumpulkan petisi 1,2 juta tanda tangan untuk diserahkan kepada Kongres Nasional pada bulan November."
3,70,"The band canceled the show at Maui's War Memorial Stadium, which was set to be attended by 9,000 people, and apologized to fans.","Band itu membatalkan pertunjukan di War Memorial Stadium Maui, yang telah disiapkan untuk dihadiri oleh 9.000 orang, lalu meminta maaf k..."
4,71,"The band's management company, HK Management Inc., gave no initial reason when they canceled on September 20, but blamed logistical reas...","Perusahaan manajemen band, HK Management Inc., awalnya tidak memberikan alasan saat membatalkan pada tanggal 20 September, kemudian memb..."


## 3. Translate with pre-trained NMT models
Both are Transformer **encoder–decoder** models with cross-attention (Bab 13).

In [4]:
hyps = {}
for key in (["dry-run"] if DRY_RUN else MODELS):
    label = key if DRY_RUN else m.MODELS[key]["label"]
    hyps[label] = m.dry_run_translate(refs) if DRY_RUN else m.translate(srcs, key, BATCH_SIZE, NUM_BEAMS)
    pd.DataFrame({"id": df.id, "source": srcs, "reference": refs, "hypothesis": hyps[label]}).to_csv(
        f"{OUT}/translations_{key.replace('/', '_')}.csv", index=False)
{k: v[:2] for k, v in hyps.items()}

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/796k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/801k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translating [nllb]:   0%|          | 0/13 [00:00<?, ?it/s]

{'MarianMT (opus-mt-en-id)': ['Penjara Abu Ghraib di Irak telah dinyalakan selama kerusuhan.',
  '"Jika itu menjadi komersial, kita harus memilikinya, itu adalah, tidak ada keberatan pada energi nuklir" kata Mr Costello.'],
 'NLLB-200 distilled 600M': ['Penjara Abu Ghraib di Irak telah dibakar selama kerusuhan.',
  '"Jika itu menjadi komersial, kita harus memilikinya. yaitu, tidak ada keberatan prinsip terhadap energi nuklir", kata Mr Costello.']}

## 4. Corpus-level metrics: BLEU-1…BLEU-4, chrF, chrF++, TER
* **BLEU-n** = BP · exp(Σ_{k=1..n} (1/n)·log p_k), where p_k is the modified k-gram precision.
* **chrF** = F-β (β=2) over character 1–6-grams (recall-weighted); **chrF++** adds word 1–2-grams.
* **TER** = (insertions + deletions + substitutions + shifts) / reference length — **lower is better**.

In [5]:
scores = {k: m.corpus_scores(h, refs, bertscore=USE_BERTSCORE) for k, h in hyps.items()}
summary = None
for k, sc in scores.items():
    t = sc.table[["metric", "score"]].rename(columns={"score": k})
    summary = t if summary is None else summary.merge(t, on="metric")
    sc.table.to_csv(f"{OUT}/corpus_scores_{k.replace(' ', '_')}.csv", index=False)
summary.round(2)

,metric,MarianMT (opus-mt-en-id),NLLB-200 distilled 600M
0,BLEU-1,65.87,69.22
1,BLEU-2,53.17,56.50
2,BLEU-3,43.72,46.81
3,BLEU-4,36.43,39.33
4,chrF,63.46,66.13
5,chrF++,61.14,63.88
6,TER,48.55,44.18


In [6]:
if BOOTSTRAP:
    ci = {k: {mt: m.bootstrap_ci(h, refs, mt, BOOTSTRAP, SEED) for mt in ("BLEU-1", "BLEU-4", "chrF", "TER")}
          for k, h in hyps.items()}
    display(pd.DataFrame([{"model": k, "metric": mt, "score": v, "CI low": lo, "CI high": hi}
                          for k, d in ci.items() for mt, (v, lo, hi) in d.items()]).round(2))
else:
    ci = None

,model,metric,score,CI low,CI high
0,MarianMT (opus-mt-en-id),BLEU-1,65.87,63.69,67.84
1,MarianMT (opus-mt-en-id),BLEU-4,36.43,33.63,39.15
2,MarianMT (opus-mt-en-id),chrF,63.46,61.44,65.13
3,MarianMT (opus-mt-en-id),TER,48.55,45.96,50.99
4,NLLB-200 distilled 600M,BLEU-1,69.22,67.05,71.07
5,NLLB-200 distilled 600M,BLEU-4,39.33,36.26,42.51
6,NLLB-200 distilled 600M,chrF,66.13,64.40,68.08
7,NLLB-200 distilled 600M,TER,44.18,41.53,46.79


## 5. Effect of n-gram length on BLEU

In [7]:
cmp_ = pd.DataFrame({k: [s.table.set_index("metric").loc[f"BLEU-{n}", "score"] for n in range(1, 5)]
                     for k, s in scores.items()}, index=[f"BLEU-{n}" for n in range(1, 5)])
cmp_["drop vs BLEU-1 (first model)"] = cmp_.iloc[:, 0] - cmp_.iloc[0, 0]
display(cmp_.round(2))
for k, s in scores.items():
    print(k, "| individual precisions:", s.precisions["precision (%)"].round(1).tolist(),
          "| BP =", round(s.precisions.attrs["bp"], 3))
fig = m.plot_ngram_effect(scores, f"{OUT}/ngram_effect.png")

,MarianMT (opus-mt-en-id),NLLB-200 distilled 600M,drop vs BLEU-1 (first model)
BLEU-1,65.87,69.22,0.00
BLEU-2,53.17,56.50,-12.70
BLEU-3,43.72,46.81,-22.15
BLEU-4,36.43,39.33,-29.44


MarianMT (opus-mt-en-id) | individual precisions: [69.0, 44.9, 31.0, 22.1] | BP = 0.955
NLLB-200 distilled 600M | individual precisions: [71.9, 47.9, 33.4, 24.2] | BP = 0.962


**Reading the curve.** Cumulative BLEU falls monotonically from n=1 to n=4: unigram matches only need the right *words*, whereas 3–4-gram matches also need the right *local order and inflection*.
English→Indonesian differs in modifier–head order, relative clauses (*yang*), affixation and tense/aspect expression, so long n-gram matches are rarer even when the translation is acceptable.

## 6. Sentence-level scores & the lowest-scoring sentences (≥ 5 required)

In [8]:
sents = {k: m.sentence_scores(srcs, h, refs, df.id.tolist()) for k, h in hyps.items()}
worst = {k: m.worst_sentences(s, WORST_K) for k, s in sents.items()}
for k in sents:
    sents[k].to_csv(f"{OUT}/sentence_scores_{k.replace(' ', '_')}.csv", index=False)
    worst[k].to_csv(f"{OUT}/worst_sentences_{k.replace(' ', '_')}.csv", index=False)
    print("=" * 30, k)
    display(worst[k][["id", "sBLEU", "sChrF", "sTER", "flags"]].round(1))

============================== MarianMT (opus-mt-en-id)


,id,sBLEU,sChrF,sTER,flags
0,375,1.5,18.7,100.0,NUMBER/UNIT mismatch; OMISSION (hypothesis much shorter)
1,355,5.3,30.6,76.0,OMISSION (hypothesis much shorter); NAMED ENTITY / term translated or lost
2,555,6.0,37.8,83.3,LEXICAL CHOICE (polysemy / synonym / lexical gap?) -> read carefully
3,1396,6.9,39.9,73.1,OMISSION (hypothesis much shorter); NAMED ENTITY / term translated or lost
4,1041,7.0,40.4,100.0,LEXICAL CHOICE (polysemy / synonym / lexical gap?) -> read carefully
5,655,7.4,41.1,68.2,OMISSION (hypothesis much shorter); NAMED ENTITY / term translated or lost
6,1107,4.5,46.3,64.3,
7,1377,8.0,42.1,61.5,OMISSION (hypothesis much shorter); NAMED ENTITY / term translated or lost
8,1495,6.5,45.5,76.7,
9,487,6.5,47.1,88.2,NAMED ENTITY / term translated or lost


============================== NLLB-200 distilled 600M


,id,sBLEU,sChrF,sTER,flags
0,355,5.6,29.7,76.0,OMISSION (hypothesis much shorter); NAMED ENTITY / term translated or lost
1,1042,5.4,44.7,73.7,LEXICAL CHOICE (polysemy / synonym / lexical gap?) -> read carefully; REDUPLICATION in reference missing in hypothesis (plurality)
2,533,6.4,43.3,71.4,LEXICAL CHOICE (polysemy / synonym / lexical gap?) -> read carefully
3,1340,8.8,29.3,72.2,OMISSION (hypothesis much shorter); REDUPLICATION in reference missing in hypothesis (plurality)
4,1199,5.6,47.8,76.0,
5,1495,5.2,49.3,66.7,
6,1570,12.0,44.5,72.0,OMISSION (hypothesis much shorter); NAMED ENTITY / term translated or lost
7,1152,9.1,47.7,82.4,NAMED ENTITY / term translated or lost
8,1808,13.2,42.2,63.2,LEXICAL CHOICE (polysemy / synonym / lexical gap?) -> read carefully
9,675,8.1,48.3,69.2,NAMED ENTITY / term translated or lost


In [9]:
# Read each triple carefully: source / reference / hypothesis
first = list(worst)[0]
for _, r in worst[first].iterrows():
    print(f"#{r['id']}  sBLEU={r['sBLEU']:.1f}  sChrF={r['sChrF']:.1f}  TER={r['sTER']:.1f}")
    print("  EN :", r["source"]); print("  REF:", r["reference"]); print("  MT :", r["hypothesis"])
    print("  auto-flags:", r["flags"] or "-"); print()

#375  sBLEU=1.5  sChrF=18.7  TER=100.0
  EN : A person who weighs 200 pounds (90kg) on Earth would weigh about 36 pounds (16kg) on Io. So the gravity, of course, pulls less on you.
  REF: Orang yang berbobot 90 kg di Bumi akan berbobot 16 kg di Io. Itu sebabnya gravitasi memiliki daya tarik lebih rendah.
  MT : Jadi gravitasi, tentu saja, mengurangi berat badan Anda.
  auto-flags: NUMBER/UNIT mismatch; OMISSION (hypothesis much shorter)

#355  sBLEU=5.3  sChrF=30.6  TER=76.0
  EN : The blade on a modern ice skate has a double edge with a concave hollow between them. The two edges allow for a better grasp of the ice, even when tilted.
  REF: Bilah pada seluncur es modern memiliki sisi ganda dengan cekungan kosong diantaranya. Dua sisi ini memungkinkan cengkeraman yang lebih baik terhadap es, bahkan ketika dimiringkan.
  MT : Pisau pada skate es modern memiliki tepi ganda dengan cekung berongga antara mereka.
  auto-flags: OMISSION (hypothesis much shorter); NAMED ENTITY / term translate

### 6.1 Manual linguistic annotation (YOUR part)
Fill the dictionary below for **at least 5** sentences, using the checklist:
*structural disparity / D-M vs M-D word order · polysemy · lexical gap · tense–aspect & voice (me-/di-) · plurality (reduplication) · pronoun/politeness · named entity & number · acceptable synonym (BLEU too harsh)*.
The key is the sentence `id` printed above.

In [10]:
import json, pathlib
annotations = {
    # "123": {"phenomenon": "Structural disparity (M-D vs D-M)",
    #         "analysis": "EN 'the newly elected committee chairman' ... ID reference puts the head noun first ... MT kept English order ..."},
}
p = pathlib.Path("report/annotations.json")
if annotations:
    p.write_text(json.dumps(annotations, ensure_ascii=False, indent=2), encoding="utf-8")
print("annotations saved:", len(annotations))

annotations saved: 0


## 7. Where BLEU fails: sentences with low BLEU but high chrF

In [11]:
for k, s in sents.items():
    ex = m.limitation_examples(s, 5)["low_bleu_ok_chrf"]
    print("=" * 30, k)
    for _, r in ex.iterrows():
        print(f"#{r['id']} sBLEU={r['sBLEU']:.1f} sChrF={r['sChrF']:.1f}\n  REF: {r['reference']}\n  MT : {r['hypothesis']}\n")

============================== MarianMT (opus-mt-en-id)
#766 sBLEU=21.5 sChrF=85.2
  REF: Keluarga dengan anak-anak merupakan pemandangan langka, tetapi beberapa hostel memperbolehkan mereka di kamar pribadi.
  MT : Keluarga dengan anak - anak merupakan pemandangan yang langka, tetapi beberapa asrama memperbolehkan mereka berada di kamar - kamar pribadi.

#136 sBLEU=11.7 sChrF=71.4
  REF: Faktor-faktor alam dapat bersinggungan untuk menghasilkan kondisi ideal sehingga jumlah alga ini meningkat secara drastis.
  MT : Faktor - faktor alami dapat berpotongan untuk menghasilkan kondisi yang ideal, sehingga alga ini dapat meningkat secara dramatis.

#352 sBLEU=15.2 sChrF=69.4
  REF: Jarum baja mengambang di atas permukaan air karena tegangan permukaan.
  MT : Jarum baja mengapung di atas air karena ketegangan permukaan.

#1552 sBLEU=25.2 sChrF=76.9
  REF: Kambing tampaknya mulai dijinakkan sekitar 10.000 tahun lalu di pegunungan Zagros di Iran.
  MT : Kambing tampaknya pertama kali dijinakk

In [12]:
for k, s in sents.items():
    m.plot_score_distribution(s, f"{OUT}/score_dist_{k.replace(' ', '_')}.png", k)
print("corr(sBLEU, sChrF):", {k: round(s["sBLEU"].corr(s["sChrF"]), 3) for k, s in sents.items()})

corr(sBLEU, sChrF): {'MarianMT (opus-mt-en-id)': np.float64(0.846), 'NLLB-200 distilled 600M': np.float64(0.863)}


## 8. Build the written report (Markdown)

In [13]:
from src.report import build_report
md = build_report(scores, sents, worst, len(df), SEED, ci, dry_run=DRY_RUN)
rp = f"{OUT}/laporan_analisis.md" if DRY_RUN else "report/laporan_analisis.md"
open(rp, "w", encoding="utf-8").write(md)
print("report written to", rp)

report written to report/laporan_analisis.md
